In [280]:
import pandas as pd
from rdkit import Chem
from dataclasses import dataclass
from typing import Optional

In [ ]:
base = "/data"
antibodies_interactions = pd.read_csv(f"{base}/row/new_antibody_interactions.csv")
aptamers_interactions = pd.read_csv(f"{base}/row/new_aptamers_interactions.csv")

In [281]:
antibodies_interactions.drop(columns=['Unnamed: 0'], inplace=True)
aptamers_interactions.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
AA20 = set("ACDEFGHIKLMNPQRSTVWY")
DNA  = set("ACGT")
RNA  = set("ACGU")
NUC  = set("ACGTU")

NUC_AMBIG = set("NRYSWKMBDHVX")
NUC_LIKE  = NUC | NUC_AMBIG

@dataclass(frozen=True)
class SeqCheck:
    kind: str                 # "DNA"|"RNA"|"NUCLEIC_MIXED"|"NUCLEIC_AMBIG"|"AA"|"SMILES"|"INVALID"
    length: int
    invalid_chars: str
    canonical: Optional[str]

def try_canonical_smiles(s: str) -> Optional[str]:
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

def classify_sequence(s: str, allow_aa_uo: bool = False) -> SeqCheck:
    if s is None:
        return SeqCheck("INVALID", 0, "None", None)

    s = s.strip()
    if not s:
        return SeqCheck("INVALID", 0, "EMPTY", None)

    s_smiles = s
    s_seq = s.upper()
    chars = set(s_seq)

    if chars <= NUC_LIKE:
        amb = chars & NUC_AMBIG
        if amb:
            return SeqCheck("INVALID", len(s_seq), "".join(sorted(amb)), None)

        # strictly A/C/G/T/U only
        has_t = "T" in chars
        has_u = "U" in chars
        if has_t and has_u:
            return SeqCheck("NUCLEIC_MIXED", len(s_seq), "", None)
        if has_t:
            return SeqCheck("DNA", len(s_seq), "", None)
        if has_u:
            return SeqCheck("RNA", len(s_seq), "", None)
        return SeqCheck("NUCLEIC_AMBIG", len(s_seq), "", None)  # only A/C/G

    # 2) AA 
    AA = AA20 | (set("UO") if allow_aa_uo else set())
    if chars <= AA:
        return SeqCheck("AA", len(s_seq), "", None)

    # 3) SMILES 
    can = try_canonical_smiles(s_smiles)
    if can is not None:
        return SeqCheck("SMILES", len(s_smiles), "", can)

    # 4) INVALID 
    allowed = AA | NUC  # intentionally not including ambiguity here
    invalid = "".join(sorted(chars - allowed))
    return SeqCheck("INVALID", len(s_seq), invalid, None)

In [283]:
antibodies_interactions

,antibody_name,antibody_sequence,target_name,target_seq
0,9cq4_H_H,QVQLQQSGAELARPGASVKMSCKASGYTFTRYTMHWVKQRPGQGLE...,t-cell surface glycoprotein cd3 epsilon chain,MQSGTHWRVLGLCLLSVGVWGQDGNEEMGGITQTPYKVSISGTTVI...
1,9cq8_C_H,QVQLQQSGAELVRPGASVTLSCKASGYTFTDYEVYWVKQTPVHGLE...,9c2 tcr delta chain,AQKVTQAQSSVSMPVRKAVTLNCLYETSWWSYYIFWYKQLPSKEMI...
2,9cq8_G_H,QVQLQQSGAELVRPGASVTLSCKASGYTFTDYEVYWVKQTPVHGLE...,9c2 tcr delta chain,AQKVTQAQSSVSMPVRKAVTLNCLYETSWWSYYIFWYKQLPSKEMI...
3,9cql_J_H,QVQLQQPGADLVRPGTSVKLSCKASGYTFTSYWMHWVQQRPGQGLE...,9c2 tcr delta chain,AQKVTQAQSSVSMPVRKAVTLNCLYETSWWSYYIFWYKQLPSKEMI...
4,9cql_C_H,QVQLQQPGADLVRPGTSVKLSCKASGYTFTSYWMHWVQQRPGQGLE...,9c2 tcr delta chain,AQKVTQAQSSVSMPVRKAVTLNCLYETSWWSYYIFWYKQLPSKEMI...
...,...,...,...,...
19865,7tas_L_L,QSVLTQPPSASGTPGQRVTISCSGSSANIGSNTVNWYQHLPGTAPK...,spike glycoprotein,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...
19866,7lo6_I_L,DIVMTQSPATLSVSPGERATLSCRASESVSSDLAWYQQKPGQAPRL...,envelope glycoprotein bg505 sosip.664 gp120,NLWVTVYYGVPVWKDAETTLFCASDAKAYETEKHNVWATHACVPTD...
19867,3vi3_L_L,DIVMTQATPSIPVTPGESVSISCRSNKSLLHSNGNTYLYWFLQRPG...,integrin beta-1,QTDENRCLKANAKSCGECIQAGPNCGWCTNSTFLQEGMPTSARCDD...
19868,6zdg_G_L,DIQMTQSPSSLSASVGDRVTITCRASQSISSYLNWYQQKPGKAPKL...,spike glycoprotein,TNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFK...


In [284]:
antibody = (
    antibodies_interactions
        [["antibody_name", "antibody_sequence"]]
        .rename(columns={
            "antibody_name": "name",
            "antibody_sequence": "content"
        })
)

target = (
    antibodies_interactions
        [["target_name", "target_seq"]]
        .rename(columns={
            "target_name": "name",
            "target_seq": "content"
        })
)

antibodies_target = pd.concat([antibody, target])

In [ ]:
antibodies_target.dropna(inplace=True)
antibodies_target.shape
res = antibodies_target["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

[17:09:20] SMILES Parse Error: syntax error while parsing: EVQLVESGGGLVQPGGSLRLSCTASGFTFSNYGFHWVRQAPGKGLEWVTIISYDGITKHYADSVKDRFTVSRDNSKTMVYLQMNNLKLDDTAVYYCARDLGTYDDSWGQGVLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGXLVKDYFPEXVTXSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKXVEPKSCDKTHTCPPCPAPELLGGPSVFLFPPKPKDTLMISRTPEVTCVVVDVSHEDPEVKFNWYVDGVEVHNAKTKPREEQYNSTYRVVSVLTVLHQDWLNGKEYKCKVSNKALPAPIEKTISKAKGQPREPQVYTLPPSRDEXTKNQVSLTCLVKGFYPSDIAVEWESNGQPENNYKTTPPVLDSDGSFFLYSKLTVDKSRWQQGNVFSCSVMHEALHNHYTQKSLSLSPGK
[17:09:20] SMILES Parse Error: check for mistakes around position 1:
[17:09:20] EVQLVESGGGLVQPGGSLRLSCTASGFTFSNYGFHWVRQAP
[17:09:20] ^
[17:09:20] SMILES Parse Error: Failed parsing SMILES 'EVQLVESGGGLVQPGGSLRLSCTASGFTFSNYGFHWVRQAPGKGLEWVTIISYDGITKHYADSVKDRFTVSRDNSKTMVYLQMNNLKLDDTAVYYCARDLGTYDDSWGQGVLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGXLVKDYFPEXVTXSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKXVEPKSCDKTHTCPPCPAPELLGGPSVFLFPPKPKDTLMISRTPEVTCVVVDVSHEDPEVKFNWYVDGVEVHNAKTKPREEQYNSTYRVVS

content
AA         39320
INVALID      410
Name: count, dtype: int64

In [287]:
mask_INVALID = res.apply(lambda x: x.kind) == "INVALID"
bad_INVALID = antibodies_target.loc[mask_INVALID, ["content", "name"]].copy()
bad_INVALID["invalid_chars"] = res[mask_INVALID].apply(lambda x: x.invalid_chars)
bad_INVALID["len"] = res[mask_INVALID].apply(lambda x: x.length)
bad_INVALID = bad_INVALID.reset_index()
bad_INVALID = bad_INVALID[["index", "content", "name", "invalid_chars", "len"]]
# bad_INVALID.to_csv("antibodies_target_INVALID.csv")
bad_INVALID

,index,content,name,invalid_chars,len
0,661,EVQLVESGGGLVQPGGSLRLSCTASGFTFSNYGFHWVRQAPGKGLE...,8q5y_R_H,X,447
1,662,EVQLVESGGGLVQPGGSLRLSCTASGFTFSNYGFHWVRQAPGKGLE...,8q5y_G_H,X,447
2,663,EVQLVESGGGLVQPGGSLRLSCTASGFTFSNYGFHWVRQAPGKGLE...,8q5y_B_H,X,447
3,1040,EVQLVESGGGLVQPGRSLRLSCAASGFPFDDYAIHWVRLAPGKGLE...,8suo_H_H,X,228
4,4126,QVQLVESGGGLIQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLE...,6xcn_H_H,X,230
...,...,...,...,...,...
405,19767,XQFHPX,peptide n-acetyl-l-gln-d-phe-l-his-d-pro-b-ala-oh,X,6
406,19772,XEHPX,peptide n-acetyl-d-glu-l-his-d-pro-nh2,X,5
407,19782,XNPNANPNANPNAX,npna3 peptide,X,14
408,19847,RENAKAKTDHGAEIVYKSPVVSGDTSPRHLX,microtubule-associated protein tau,X,31


In [288]:
bad_INVALID['invalid_chars'].value_counts()

invalid_chars
X         369
DS          8
KS          8
K           6
M           4
HKNVX       4
S           4
DKY         2
DHMNW       2
DHMRWX      2
H           1
Name: count, dtype: int64

In [290]:
mask = ~bad_INVALID["invalid_chars"].isin(["X"])

subset = bad_INVALID[mask]
rest   = bad_INVALID[~mask]

In [291]:
rest['invalid_chars'].value_counts()

invalid_chars
X    369
Name: count, dtype: int64

In [292]:
rest.to_csv("antibodies_target_INVALID_X.csv")

In [295]:
mask_AA = res.apply(lambda x: x.kind) == "AA"
bad_AA = antibodies_target.loc[mask_AA, ["content", "name"]].copy()
bad_AA["invalid_chars"] = res[mask_AA].apply(lambda x: x.invalid_chars)
bad_AA["len"] = res[mask_AA].apply(lambda x: x.length)
bad_AA = bad_AA.reset_index()
bad_AA = bad_AA[["index", "content", "name", "invalid_chars", "len"]]
bad_AA
# bad_AA.to_csv("antibodies_target_AA.csv")

,index,content,name,invalid_chars,len
0,0,QVQLQQSGAELARPGASVKMSCKASGYTFTRYTMHWVKQRPGQGLE...,9cq4_H_H,,238
1,1,QVQLQQSGAELVRPGASVTLSCKASGYTFTDYEVYWVKQTPVHGLE...,9cq8_C_H,,225
2,2,QVQLQQSGAELVRPGASVTLSCKASGYTFTDYEVYWVKQTPVHGLE...,9cq8_G_H,,225
3,3,QVQLQQPGADLVRPGTSVKLSCKASGYTFTSYWMHWVQQRPGQGLE...,9cql_J_H,,224
4,4,QVQLQQPGADLVRPGTSVKLSCKASGYTFTSYWMHWVQQRPGQGLE...,9cql_C_H,,224
...,...,...,...,...,...
39315,19865,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,spike glycoprotein,,1288
39316,19866,NLWVTVYYGVPVWKDAETTLFCASDAKAYETEKHNVWATHACVPTD...,envelope glycoprotein bg505 sosip.664 gp120,,479
39317,19867,QTDENRCLKANAKSCGECIQAGPNCGWCTNSTFLQEGMPTSARCDD...,integrin beta-1,,454
39318,19868,TNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFK...,spike glycoprotein,,194


In [296]:
bad_AA = pd.concat([bad_AA, subset])
bad_AA.to_csv("antibodies_target_AA.csv")
bad_AA

,index,content,name,invalid_chars,len
0,0,QVQLQQSGAELARPGASVKMSCKASGYTFTRYTMHWVKQRPGQGLE...,9cq4_H_H,,238
1,1,QVQLQQSGAELVRPGASVTLSCKASGYTFTDYEVYWVKQTPVHGLE...,9cq8_C_H,,225
2,2,QVQLQQSGAELVRPGASVTLSCKASGYTFTDYEVYWVKQTPVHGLE...,9cq8_G_H,,225
3,3,QVQLQQPGADLVRPGTSVKLSCKASGYTFTSYWMHWVQQRPGQGLE...,9cql_J_H,,224
4,4,QVQLQQPGADLVRPGTSVKLSCKASGYTFTSYWMHWVQQRPGQGLE...,9cql_C_H,,224
...,...,...,...,...,...
370,18432,GKGSG,peptide,KS,5
371,18468,XHVVVNNKVATHX,class 1 outer membrane protein variable region 2,HKNVX,13
379,18738,TGHRMAWDMMMX,hcv e1 peptide,DHMRWX,12
386,19116,GKGSG,peptide,KS,5


In [297]:
aptamers_interactions

,apt_seq,target_name,target_seq,apt_name,Entry_ID,Molecule_ID,Target_RNA_ID,pKd,pdb,res,...,Binding Conditions/Buffer,Binding Temp,Specificity,Comments,Sequence with modifications,Length,Molecular Weight,Extinction Coefficient,URL,kd_value
0,UCGGAGGUGGUUCAGCUCUGCAUCGACAGUUGGC,dsRNA_activated_protein_kinase,MDSTNYISKLFEYAQRQGQISDIKFEEVGTDGPDHLKTFTLRVVIK...,APIPred_1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CGGGGTGTTGTCCTGTGCTCTCCGGAGAGCACAGGACAACACCCCG,Human_Nuclear_Factor_kB_RelA_p65,MDELFPLIFPAEPAQASGPYVEIIEQPKQRGMRFRYKCEGRSAGSI...,APIPred_2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GCTGCAGTTGCACTGAATTCGCCTCTCGCCTCCGTACACTTAGTCG...,atSPL14,MDEVGAQVAAPMFIHQSLGRKRDLYYPMSNRLVQSQPQRRDEWNSK...,APIPred_3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TATTTGCCCTTGCAGGCCGCAGGAGTGCTAGCAGT,HIV-1-RT,PISPIDTVPVKLKPGMDGPKVKQWPLTEEKIKALTEICNEMEKEGK...,APIPred_4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,UCUCUGGGCUCUUAGGAGAACGGAUAGGAGUGUGCUCGCU,prothrombin,LQENFLPQPRQQHHGTLVLHYRPHREEAGMQHPCLWPGSSHCSDDS...,APIPred_5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1921,GACGACCATTGGTCTCTTTAGAGGTGTCCGTTAGGTCGTC,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2,Aptamer_ds_443,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active
1922,GACGACGTTGTGGTCTATTCATAGGTGTCTGCATCGTCGTC,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2,Aptamer_ds_444,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active
1923,GACGACGGTACAGGTCTATTCATAGGTGTCCGCAACGTCGTC,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2,Aptamer_ds_445,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active
1924,GACGACCAGTGAGGTCTATTCATAGGTGTCCGCCAGGTCGTC,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2,Aptamer_ds_446,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active


In [298]:
aptamers = (
    aptamers_interactions
        [["apt_name", "apt_seq"]]
        .rename(columns={
            "apt_name": "name",
            "apt_seq": "content"
        })
)

target = (
    aptamers_interactions
        [["target_name", "target_seq"]]
        .rename(columns={
            "target_name": "name",
            "target_seq": "content"
        })
)

aptamers_target = pd.concat([aptamers, target])

In [299]:
aptamers_target

,name,content
0,APIPred_1,UCGGAGGUGGUUCAGCUCUGCAUCGACAGUUGGC
1,APIPred_2,CGGGGTGTTGTCCTGTGCTCTCCGGAGAGCACAGGACAACACCCCG
2,APIPred_3,GCTGCAGTTGCACTGAATTCGCCTCTCGCCTCCGTACACTTAGTCG...
3,APIPred_4,TATTTGCCCTTGCAGGCCGCAGGAGTGCTAGCAGT
4,APIPred_5,UCUCUGGGCUCUUAGGAGAACGGAUAGGAGUGUGCUCGCU
...,...,...
1921,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2
1922,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2
1923,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2
1924,Theophylline,CN1C2=C(C(=O)N(C1=O)C)NC=N2


In [ ]:
aptamers_target.dropna(inplace=True)
aptamers_target.shape
res = aptamers_target["content"].apply(classify_sequence)

kinds = res.apply(lambda x: x.kind).value_counts()
kinds

[17:51:32] SMILES Parse Error: syntax error while parsing: TTGACCGCT-GTGTGACGCAACACTCAATTTCTTCCAGCCGGTCCG
[17:51:32] SMILES Parse Error: check for mistakes around position 1:
[17:51:32] TTGACCGCT-GTGTGACGCAACACTCAATTTCTTCCAGCCG
[17:51:32] ^
[17:51:32] SMILES Parse Error: Failed parsing SMILES 'TTGACCGCT-GTGTGACGCAACACTCAATTTCTTCCAGCCGGTCCG' for input: 'TTGACCGCT-GTGTGACGCAACACTCAATTTCTTCCAGCCGGTCCG'
[17:51:32] SMILES Parse Error: syntax error while parsing: PISPIDTVPVKLKPGMDGPKVKQWPLTEEKIKALTEICNEMEKEGKISKIGPENPYNTPVFAIKKKDSNKWRKLVDFRELNKRTQDFWEVQLGIPHPAGLKKKKSVTVLDVGDAYFSVPLDESFRKYTAFTIPSINNETPGVRYQYNVLPQGWKGSPAIFQCSMTKILEPXRXKNPEMVIYQYMDDLYVGSDLEIGQHRAKIEELRSHLLSWGFTTPDKKHQKEPPFLWMGYELHPDSWTVQPIELPEKDSWTVNE
[17:51:32] SMILES Parse Error: check for mistakes around position 6:
[17:51:32] PISPIDTVPVKLKPGMDGPKVKQWPLTEEKIKALTEICNEM
[17:51:32] ~~~~~^
[17:51:32] SMILES Parse Error: Failed parsing SMILES 'PISPIDTVPVKLKPGMDGPKVKQWPLTEEKIKALTEICNEMEKEGKISKIGPENPYNTPVFAIKKKDSNKWRKLVDFRELNKRTQDF

content
SMILES           1169
RNA              1042
DNA               700
AA                696
INVALID           159
NUCLEIC_MIXED      23
NUCLEIC_AMBIG      17
Name: count, dtype: int64

In [309]:
mask_INVALID = res.apply(lambda x: x.kind) == "INVALID"
INVALID_rows = aptamers_target.loc[mask_INVALID, ["content", "name"]].copy()
INVALID_rows["invalid_chars"] = res[mask_INVALID].apply(lambda x: x.invalid_chars)
INVALID_rows["len"] = res[mask_INVALID].apply(lambda x: x.length)
INVALID_rows = INVALID_rows.reset_index()
INVALID_rows = INVALID_rows[["index", "content", "name", "invalid_chars", "len"]]
INVALID_rows.to_csv("INVALID_from_aptamers_interactions.csv", index=False)
INVALID_rows

,index,content,name,invalid_chars,len
0,108,BCGAGGCGTGTGCAAGGCGTGTTGTTCAAAGGTGTGTGTGT,APIPred_109,B,41
1,109,GTCAAGNACGCGAACCCCGCATTCCCAAGAACGGCAACCCT,APIPred_110,N,41
2,189,NGACUGAUUUUUCCUUGNCAGUGUAAUUUCCUGGCUGCCC,APIPred_190,N,40
3,469,BAGCGCCGGTGGTGGTGGGGGTTGTTGTTACGCGGTATTAT,APIPred_469,B,41
4,492,UUGUAAGCUCAACGGGUGGAGGGANCGGGCGAACGGGCUGGAGCAC...,APIPred_492,N,66
...,...,...,...,...,...
154,531,WPLTEEKIKALVEICXEMEKEGKISKIGPENPYNTPVFAIKKKDST...,HIV-RT,X,213
155,562,PISPIDTVPVKLKPGMDGPKVKQWPLTEEKIKALTEICNEMEKEGK...,M3-HIV-1-RT,X,256
156,563,WPLTEEKIKALVEICXEMEKEGKISKIGPENPYNTPVFAIKKKDST...,HIV-RT,X,213
157,618,PISPIDTVPVKLKPGMDGPKVKQWPLTEEKIKALTEICNEMEKEGK...,HIV-1-RT,X,256


In [308]:
INVALID_rows['invalid_chars'].value_counts()

invalid_chars
XY    56
X     46
N     35
D      9
M      9
B      2
-      1
DX     1
Name: count, dtype: int64

In [303]:
INVALID_rows.iloc[-1]['content'], INVALID_rows.iloc[-3]['content']

('Coc1ccc2c(c1)c(Nc1ccc(c(c1)CN1CCN(CC1)C)O)c1c(n2)cc(cc1)Cl',
 '[C@H]1([C@@H]([C@H](O)[C@H]([C@H](C1)N#N)O)OCc1ccc(cc1)c1ccc(cc1)CO[C@H]1[C@H](C[C@@H]([C@@H](O)[C@@H]1O)N#N)N#N)N#N')

In [304]:
old = 'Coc1ccc2c(c1)c(Nc1ccc(c(c1)CN1CCN(CC1)C)O)c1c(n2)cc(cc1)Cl'
new = 'COc1ccc2c(c1)c(Nc1ccc(c(c1)CN1CCN(CC1)C)O)c1c(n2)cc(cc1)Cl'

mask = aptamers_target["content"].eq(old)

aptamers_target.loc[mask, "content"] = new

In [305]:
old = '[C@H]1([C@@H]([C@H](O)[C@H]([C@H](C1)N#N)O)OCc1ccc(cc1)c1ccc(cc1)CO[C@H]1[C@H](C[C@@H]([C@@H](O)[C@@H]1O)N#N)N#N)N#N'
new = '[C@H]1([C@@H]([C@H](O)[C@H]([C@H](C1)N=[N+]=[N-])O)OCc1ccc(cc1)c1ccc(cc1)CO[C@H]1[C@H](C[C@@H]([C@@H](O)[C@@H]1O)N=[N+]=[N-])N=[N+]=[N-])N=[N+]=[N-]'

mask = aptamers_target["content"].eq(old)

aptamers_target.loc[mask, "content"] = new

In [310]:
mask_RNA = res.apply(lambda x: x.kind) == "RNA"
RNA_rows = aptamers_target.loc[mask_RNA, ["content", "name"]].copy()
RNA_rows["invalid_chars"] = res[mask_RNA].apply(lambda x: x.invalid_chars)
RNA_rows["len"] = res[mask_RNA].apply(lambda x: x.length)
RNA_rows = RNA_rows.reset_index()
RNA_rows = RNA_rows[["index", "content", "name", "invalid_chars", "len"]]
RNA_rows.to_csv("RNA_from_aptamers_interactions.csv")
RNA_rows

,index,content,name,invalid_chars,len
0,0,UCGGAGGUGGUUCAGCUCUGCAUCGACAGUUGGC,APIPred_1,,34
1,4,UCUCUGGGCUCUUAGGAGAACGGAUAGGAGUGUGCUCGCU,APIPred_5,,40
2,5,GGGAGACAAGAAUAAACGCUCAACACAGAACGCGGUCCCCACACAG...,APIPred_6,,88
3,7,CACCUACUAGACCACUUUUUGAGCCGGUUUUUUCGGGAACUUGCCAA,APIPred_8,,47
4,8,GCAACAGAGGCUGAUGGUAGACGCCGGCCA,APIPred_9,,30
...,...,...,...,...,...
1037,1909,GGUACUUCGGGAAG,Aptamer_ds_435,,14
1038,1910,GGCGAUACCAGCCGAAAGGCCCUUGGCAGCGUC,Aptamer_ds_268,,33
1039,1911,CCAGAAACCUUGGCA,Aptamer_ds_436,,15
1040,1912,CCGAAACUUGGCA,Aptamer_ds_437,,13


In [ ]:
mask_DNA = res.apply(lambda x: x.kind) == "DNA"
DNA_rows = aptamers_target.loc[mask_DNA, ["content", "name"]].copy()
DNA_rows["invalid_chars"] = res[mask_DNA].apply(lambda x: x.invalid_chars)
DNA_rows["len"] = res[mask_DNA].apply(lambda x: x.length)
DNA_rows = DNA_rows.reset_index()
DNA_rows = DNA_rows[["index", "content", "name", "invalid_chars", "len"]]
DNA_rows.to_csv("DNA_from_aptamers_interactions.csv")
DNA_rows

,index,content,name,invalid_chars,len
0,1,CGGGGTGTTGTCCTGTGCTCTCCGGAGAGCACAGGACAACACCCCG,APIPred_2,,46
1,2,GCTGCAGTTGCACTGAATTCGCCTCTCGCCTCCGTACACTTAGTCG...,APIPred_3,,69
2,3,TATTTGCCCTTGCAGGCCGCAGGAGTGCTAGCAGT,APIPred_4,,35
3,6,CTTCTGCCCGCCTCCTTCCGTTTTCCTGCGGCAGCGTTTTGTTGCT...,APIPred_7,,80
4,11,CGTGTCGTATTGAGTGATATGATGCACATA,APIPred_12,,30
...,...,...,...,...,...
695,1921,GACGACCATTGGTCTCTTTAGAGGTGTCCGTTAGGTCGTC,Aptamer_ds_443,,40
696,1922,GACGACGTTGTGGTCTATTCATAGGTGTCTGCATCGTCGTC,Aptamer_ds_444,,41
697,1923,GACGACGGTACAGGTCTATTCATAGGTGTCCGCAACGTCGTC,Aptamer_ds_445,,42
698,1924,GACGACCAGTGAGGTCTATTCATAGGTGTCCGCCAGGTCGTC,Aptamer_ds_446,,42


In [312]:
mask_NUCLEIC_MIXED = res.apply(lambda x: x.kind) == "NUCLEIC_MIXED"
NUCLEIC_MIXED_rows = aptamers_target.loc[mask_NUCLEIC_MIXED, ["content", "name"]].copy()
NUCLEIC_MIXED_rows["invalid_chars"] = res[mask_NUCLEIC_MIXED].apply(lambda x: x.invalid_chars)
NUCLEIC_MIXED_rows["len"] = res[mask_NUCLEIC_MIXED].apply(lambda x: x.length)
NUCLEIC_MIXED_rows = NUCLEIC_MIXED_rows.reset_index()
NUCLEIC_MIXED_rows = NUCLEIC_MIXED_rows[["index", "content", "name", "invalid_chars", "len"]]
NUCLEIC_MIXED_rows.to_csv("NUCLEIC_MIXED_from_aptamers_interactions.csv")
NUCLEIC_MIXED_rows

,index,content,name,invalid_chars,len
0,19,GCGUGCAGUGCCUUCGGCCGTGCGGTGCCUCCGUCACGCT,APIPred_20,,40
1,22,GGGAGAAGGGAAGTAACAGGGUGGGUCUCCGCAUUUGUGCUUCGAU...,APIPred_23,,72
2,43,GGGAGAAGGGAAGTAACAGGUGGGCGCGUGAUUAUGUUUGUACGAA...,APIPred_44,,73
3,49,GGGACACAATGGACGUCCGCGGCGCCAUCUCAUGUUUAGUUGUCCU...,APIPred_50,,73
4,157,GGGAGAAGGGAAGTAACAGGCUAGAAGUUUAGACCAAGGGGCGGGA...,APIPred_158,,72
5,159,AGTAATACGACTCACTATAGGGAGAATTCCGACCAGAAGGGUUAGC...,APIPred_160,,93
6,418,GGGAGAAGGGAAGTAACAGGAGCAUUAGCUACCAAGGGGUCGGGUU...,APIPred_418,,73
7,520,GGGACACAATGGACGUGCGCATAGGAAUUGCUCUCAAAAAACUGGA...,APIPred_520,,105
8,634,TAGGGAATTTCGACGGATCCAGCUGAUGUGGGACGUCAGUUAUAGG...,APIPred_634,,101
9,692,GGGAGAAGGGAAGTAACAGGUUUCGUGCCUUUUCUCGAUGGAGUGA...,APIPred_691,,65


In [313]:
mask_NUCLEIC_AMBIG = res.apply(lambda x: x.kind) == "NUCLEIC_AMBIG"
NUCLEIC_AMBIG_rows = aptamers_target.loc[mask_NUCLEIC_AMBIG, ["content", "name"]].copy()
NUCLEIC_AMBIG_rows["invalid_chars"] = res[mask_NUCLEIC_AMBIG].apply(lambda x: x.invalid_chars)
NUCLEIC_AMBIG_rows["len"] = res[mask_NUCLEIC_AMBIG].apply(lambda x: x.length)
NUCLEIC_AMBIG_rows = NUCLEIC_AMBIG_rows.reset_index()
NUCLEIC_AMBIG_rows = NUCLEIC_AMBIG_rows[["index", "content", "name", "invalid_chars", "len"]]
NUCLEIC_AMBIG_rows.to_csv("NUCLEIC_AMBIG_from_aptamers_interactions.csv")
NUCLEIC_AMBIG_rows

,index,content,name,invalid_chars,len
0,60,AAAAAAAAAAAAAAA,APIPred_61,,15
1,407,AAAAAGGAGGAGGAGGA,APIPred_407,,17
2,737,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,ss_PolyA,,200
3,921,GCGCGCGCGAAAGCGCGCGC,GC Stem Loop,,20
4,928,GGGGCCGGGGCCGGGGCCGGGGCCGGGGCCGGGGCCGGGGCCGGGG...,G4C2 repeat G-quadruplex RNA,,216
5,929,GGGGCCGGGGCCGGGGCCGGGGCCGGGGCCGGGGCCGGGGCCGGGG...,G4C2 repeat G-quadruplex RNA,,216
6,938,GCGGCGGCGGAGGCA,KRAS RNA G-quadruplex utr-1,,15
7,942,GCGGCGGCGGAGGCA,KRAS RNA G-quadruplex utr-1,,15
8,1067,CGCGCGAAAGCGCG,RNA hairpin loop I,,14
9,1068,CGCGCGAAAGCGCG,RNA hairpin loop I,,14


In [315]:
mask_AA = res.apply(lambda x: x.kind) == "AA"
AA_rows = aptamers_target.loc[mask_AA, ["content", "name"]].copy()
AA_rows["invalid_chars"] = res[mask_AA].apply(lambda x: x.invalid_chars)
AA_rows["len"] = res[mask_AA].apply(lambda x: x.length)
AA_rows = AA_rows.reset_index()
AA_rows = AA_rows[["index", "content", "name", "invalid_chars", "len"]]
AA_rows.to_csv("AA_from_aptamers_interactions_bad_rows.csv")
AA_rows

,index,content,name,invalid_chars,len
0,1482,CCTIGGGGAGTATTGCGGAGGAAGG,Aptamer_ds_95,,25
1,1486,CCTGIGGGAGTATTGCGGAGGAAGG,Aptamer_ds_99,,25
2,1490,CCTGGGIGAGTATTGCGGAGGAAGG,Aptamer_ds_103,,25
3,1494,CCTGGGGIAGTATTGCGGAGGAAGG,Aptamer_ds_107,,25
4,1498,CCTGGGGGAGTATTGCIGAGGAAGG,Aptamer_ds_111,,25
...,...,...,...,...,...
691,706,MAGRSGDSDEELLKAVRLIKFLYQSNPPPNPEGTRQARRNRRRRWR...,HIV-1_Rev_binding_protein,,116
692,707,VENLMVQELENFNPPFKLCLHKRDFIPGKWIIDNIIDSIEKSHKTV...,TLR2,,123
693,708,MKKRKVLIPLMALSTILVSSTGNLEVIQAEVKQENRLLNESESSSQ...,Anthrax_Protective_Antigen,,764
694,709,MAIPKELLVLLLGIAILSKVAECAAFGSRRGLVETVFDVTRFGARP...,NTS_1,,401


In [317]:
AA_rows[:14]

,index,content,name,invalid_chars,len
0,1482,CCTIGGGGAGTATTGCGGAGGAAGG,Aptamer_ds_95,,25
1,1486,CCTGIGGGAGTATTGCGGAGGAAGG,Aptamer_ds_99,,25
2,1490,CCTGGGIGAGTATTGCGGAGGAAGG,Aptamer_ds_103,,25
3,1494,CCTGGGGIAGTATTGCGGAGGAAGG,Aptamer_ds_107,,25
4,1498,CCTGGGGGAGTATTGCIGAGGAAGG,Aptamer_ds_111,,25
5,1502,CCTGGGGGAGTATTGCGIAGGAAGG,Aptamer_ds_115,,25
6,1506,CCTGGGGGAGTATTGCGGAIGAAGG,Aptamer_ds_119,,25
7,1510,CCTGGGGGAGTATTGCGGAGIAAGG,Aptamer_ds_123,,25
8,1518,CCTGGGGGAGTATTGCGGEGGAAGG,Aptamer_ds_131,,25
9,1525,CCTGGGGGEGTATTGCGGAGGAAGG,Aptamer_ds_138,,25


In [319]:
mask_SMILES = res.apply(lambda x: x.kind) == "SMILES"
SMILES_rows = aptamers_target.loc[mask_SMILES, ["content", "name"]].copy()
SMILES_rows["invalid_chars"] = res[mask_SMILES].apply(lambda x: x.invalid_chars)
SMILES_rows["len"] = res[mask_SMILES].apply(lambda x: x.length)
SMILES_rows = SMILES_rows.reset_index()
SMILES_rows = SMILES_rows[["index", "content", "name", "invalid_chars", "len"]]
SMILES_rows.to_csv("SMILES_from_aptamers_interactions.csv")
SMILES_rows

,index,content,name,invalid_chars,len
0,711,C1=CC=C2C(=C1)C(=O)C3=C(C2=O)C(=C(C=C3NC4=CC(=...,Cibacron blue,,117
1,712,C1=CC=C2C(=C1)C(=O)C3=C(C2=O)C(=C(C=C3NC4=CC(=...,Reactive Blue 4,,97
2,713,C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)COP(=O)(O)OP(...,ATP,,68
3,714,C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)COP(=O)(O)O)O...,AMP,,50
4,715,CC1=CC2=C(C=C1C)N(C3=NC(=O)NC(=O)C3=N2)CC(C(C(...,FAD,,108
...,...,...,...,...,...
1164,1921,CN1C2=C(C(=O)N(C1=O)C)NC=N2,Theophylline,,27
1165,1922,CN1C2=C(C(=O)N(C1=O)C)NC=N2,Theophylline,,27
1166,1923,CN1C2=C(C(=O)N(C1=O)C)NC=N2,Theophylline,,27
1167,1924,CN1C2=C(C(=O)N(C1=O)C)NC=N2,Theophylline,,27
